In [1]:
import requests
import json
import pandas as pd
from datetime import datetime
import zipfile
import io

# Define your StatCan table ID
table_id = "35100180"

# Step 1: Get the download link for the latest data
api_url = f"https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/{table_id}/en"


r = requests.get(api_url)
r.raise_for_status()

data = r.json()
zip_url = data["object"]   # <-- direct download link to ZIP file
print("✅ ZIP download URL:", zip_url)

# Step 2: Download and extract CSV in memory
response = requests.get(zip_url)
z = zipfile.ZipFile(io.BytesIO(response.content))

# List all files inside the ZIP
print("Files in ZIP:", z.namelist())

# Step 3: Read the first CSV file directly into a DataFrame
csv_filename = [f for f in z.namelist() if f.endswith(".csv")][0]
csv_metadata=[f for f in z.namelist() if f.endswith(".csv")][1]
with z.open(csv_filename) as f:
    df = pd.read_csv(f)
    


StatementMeta(, cb15abd7-fb81-4457-b705-f707249d5319, 3, Finished, Available, Finished)

✅ ZIP download URL: https://www150.statcan.gc.ca/n1/tbl/csv/35100180-eng.zip
Files in ZIP: ['35100180.csv', '35100180_MetaData.csv']


In [3]:
display(csv_metadata)

StatementMeta(, cb15abd7-fb81-4457-b705-f707249d5319, 5, Finished, Available, Finished)

'35100180_MetaData.csv'

### Filtering 2020 to latest dataset

In [ ]:
df["REF_DATE"] = pd.to_numeric(df["REF_DATE"], errors="coerce").astype("Int64")
latest_year = df["REF_DATE"].max()
df_filtered = df[df["REF_DATE"].between(2020, latest_year)]
print("✅ Filtered years:", df_filtered["REF_DATE"].unique())


StatementMeta(, 88322d23-fe0d-4739-8356-13b8faa6bdcb, 20, Finished, Available, Finished)

✅ Filtered years: <IntegerArray>
[2020, 2021, 2022, 2023, 2024]
Length: 5, dtype: Int64


In [ ]:

save_path = f"abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Files/Raw/crime_data.csv"
df_filtered.to_csv(save_path, index=False)
print(f"✅ Saved extracted CSV to {save_path}")

StatementMeta(, 88322d23-fe0d-4739-8356-13b8faa6bdcb, 21, Finished, Available, Finished)

✅ Saved extracted CSV to abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Files/Raw/crime_data.csv


### Saving the Metadata for Dimension 2
##### Finding the parent name from the Parent ID

In [14]:
with z.open(csv_metadata) as f:
    df_meta = pd.read_csv(f, skiprows=8, nrows=771)

# Filter only dimension 2
    df_dim2 = df_meta[df_meta["Dimension ID"] == '2'].copy()
    
# Create a lookup map from Member ID → Member Name
    id_to_name = dict(zip(df_dim2["Member ID"], df_dim2["Member Name"]))

# Map Parent Member ID to Parent Name
    df_dim2["Parent Name"] = df_dim2["Parent Member ID"].map(id_to_name)
    df_dim3 = df_dim2.drop(["Terminated", "Member Notes", "Member Definitions"], axis=1)

    

StatementMeta(, cb15abd7-fb81-4457-b705-f707249d5319, 16, Finished, Available, Finished)

In [15]:
save_path = f"abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Files/Raw/crime_data_metadata.csv"
df_dim3.to_csv(save_path, index=False)
print(f"Metadata Saved extracted CSV to {save_path}")

StatementMeta(, cb15abd7-fb81-4457-b705-f707249d5319, 17, Finished, Available, Finished)

Metadata Saved extracted CSV to abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Files/Raw/crime_data_metadata.csv
